<a href="https://colab.research.google.com/github/AkhileshSR/AkhileshSR/blob/main/20260922_FPI_18_RAG_Session3_RAG_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai sentence-transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 9.5 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI API key configured successfully.")

Enter your OpenAI API Key: ··········
OpenAI API key configured successfully.


In [ ]:
from llama_index.core import (
    Document,
    VectorStoreIndex,
    Settings
)

from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser
)

from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

from sentence_transformers import CrossEncoder

import pandas as pd

In [ ]:
DOCUMENT_TEXT = """
ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or two days of sick leave, a medical certificate is normally not required.
If an employee takes more than two consecutive days of sick leave,
the employee must submit a medical certificate.
Unused sick leave cannot be carried forward to the next calendar year.

Remote Work:
Employees may work remotely for up to three days per week.
Remote work requires approval from the employee's reporting manager.
Employees working remotely must remain available during core working hours
from 10 AM to 4 PM.
Employees may temporarily work remotely for five days per week when approved
by both their manager and department head.

Parental Leave:
Employees who have completed at least twelve months of continuous service
are eligible for parental leave.
Primary caregivers receive 16 weeks of paid parental leave.
Secondary caregivers receive 4 weeks of paid parental leave.
Parental leave should normally be taken within six months of the child's birth
or adoption.

Learning and Development:
Every employee receives an annual learning budget of ₹50,000.
The budget can be used for professional certifications, technical courses,
books, conferences, and approved training programs.
Any single learning expense above ₹20,000 requires manager approval.
Expenses above ₹40,000 require approval from both the manager
and department head.

Business Travel:
Employees travelling for approved business purposes can claim transportation,
hotel accommodation and meals.
Domestic hotel reimbursement is limited to ₹6,000 per night.
International hotel reimbursement is limited to ₹12,000 per night.
Meal expenses are reimbursed up to ₹1,500 per day for domestic travel
and ₹3,000 per day for international travel.

Expense Claims:
Employees must submit expense claims within 30 days of incurring the expense.
Claims submitted after 30 days require manager approval.
Individual expenses above ₹25,000 require supporting documentation
and approval from the department head.
Fraudulent expense claims may result in disciplinary action.

Performance Reviews:
Formal performance reviews are conducted twice every year.
Employees are evaluated on delivery, collaboration, technical capability,
customer impact and professional development.
Performance ratings influence annual bonuses and promotion decisions.

Promotion:
Employees are normally considered for promotion after completing
at least 18 months in their current role.
Promotion decisions consider performance history, skills,
business requirements and demonstrated ability to operate at the next level.
Exceptional employees may be considered earlier with leadership approval.
"""

document = Document(text=DOCUMENT_TEXT)

print("Document loaded.")
print()
print(DOCUMENT_TEXT[:1000])

Document loaded.


ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or two days of sick leave, a medical certificate is normally not required.
If an employee takes more than two consecutive days of sick leave,
the employee must submit a medical certificate.
Unused sick leave cannot be carried forward to the next calendar year.

Remote Work:
Employees may work remotely for up to three days per week.
Remote work requires approval from the employee's reporting manager.
Employees working remotely must remain available during core working hours
from 10

In [ ]:
print("Choose OpenAI Embedding Model")
print()
print("1. text-embedding-3-small")
print("2. text-embedding-3-large")

choice = input("\nEnter choice (1 or 2): ")

if choice == "1":
    embedding_model_name = "text-embedding-3-small"

elif choice == "2":
    embedding_model_name = "text-embedding-3-large"

else:
    raise ValueError("Please choose 1 or 2.")

embed_model = OpenAIEmbedding(
    model=embedding_model_name
)

print("\nSelected embedding model:")
print(embedding_model_name)

Choose OpenAI Embedding Model

1. text-embedding-3-small
2. text-embedding-3-large

Enter choice (1 or 2): 1

Selected embedding model:
text-embedding-3-small


In [ ]:
print("CHUNKING STRATEGY")
print("=================")
print("1. Sentence/Token Chunking")
print("2. Semantic Chunking")

chunk_choice = input("\nChoose chunking strategy (1 or 2): ")

if chunk_choice == "1":

    chunk_size = int(
        input("Enter chunk size in tokens (try 100, 200, 400): ")
    )

    chunk_overlap = int(
        input("Enter chunk overlap in tokens (try 0, 20, 50): ")
    )

    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    nodes = splitter.get_nodes_from_documents([document])

    chunking_strategy = "Sentence/Token Chunking"

elif chunk_choice == "2":

    breakpoint_percentile = int(
        input(
            "Enter semantic breakpoint percentile "
            "(try 80, 90, 95): "
        )
    )

    splitter = SemanticSplitterNodeParser(
        buffer_size=1,
        breakpoint_percentile_threshold=breakpoint_percentile,
        embed_model=embed_model
    )

    nodes = splitter.get_nodes_from_documents([document])

    chunking_strategy = "Semantic Chunking"

else:
    raise ValueError("Please select 1 or 2.")

print("\nChunking complete!")
print("Strategy:", chunking_strategy)
print("Number of chunks:", len(nodes))

CHUNKING STRATEGY
1. Sentence/Token Chunking
2. Semantic Chunking

Choose chunking strategy (1 or 2): 1
Enter chunk size in tokens (try 100, 200, 400): 200
Enter chunk overlap in tokens (try 0, 20, 50): 20

Chunking complete!
Strategy: Sentence/Token Chunking
Number of chunks: 4


In [ ]:
print(f"TOTAL CHUNKS: {len(nodes)}")
print("=" * 80)

for i, node in enumerate(nodes):

    text = node.get_content()

    print(f"\nCHUNK {i+1}")
    print(f"Characters: {len(text)}")
    print("-" * 80)
    print(text)
    print("=" * 80)

TOTAL CHUNKS: 4

CHUNK 1
Characters: 993
--------------------------------------------------------------------------------
ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or two days of sick leave, a medical certificate is normally not required.
If an employee takes more than two consecutive days of sick leave,
the employee must submit a medical certificate.
Unused sick leave cannot be carried forward to the next calendar year.

Remote Work:
Employees may work remotely for up to three days per week.
Remote work requires approval from the employee'

In [ ]:
Settings.embed_model = embed_model

index = VectorStoreIndex(
    nodes,
    embed_model=embed_model
)

print("Vector index created successfully.")
print()
print("Embedding model:", embedding_model_name)
print("Chunks indexed:", len(nodes))

Vector index created successfully.

Embedding model: text-embedding-3-small
Chunks indexed: 4


In [ ]:
question = input("Enter your question: ")

top_k = int(
    input("How many chunks should we retrieve? Top-K (try 1, 2, 3, 5): ")
)

retriever = index.as_retriever(
    similarity_top_k=top_k
)

retrieved_nodes = retriever.retrieve(question)

print("\nQUESTION:")
print(question)

print("\nTOP-K:", top_k)

print("\nRETRIEVED CHUNKS")
print("=" * 80)

for rank, result in enumerate(retrieved_nodes, start=1):

    print(f"\nRANK {rank}")
    print(f"Similarity Score: {result.score:.4f}")
    print("-" * 80)
    print(result.node.get_content())
    print("=" * 80)

Enter your question: how many annual leave do i have
How many chunks should we retrieve? Top-K (try 1, 2, 3, 5): 5

QUESTION:
how many annual leave do i have

TOP-K: 5

RETRIEVED CHUNKS

RANK 1
Similarity Score: 0.5726
--------------------------------------------------------------------------------
ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or two days of sick leave, a medical certificate is normally not required.
If an employee takes more than two consecutive days of sick leave,
the employee must submit a medical certificate.
Unused sick le

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading Cross-Encoder...")

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-Encoder loaded successfully.")

Loading Cross-Encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded successfully.


In [ ]:
print("RERANKING")
print("=" * 60)

use_reranking = input(
    "Use Cross-Encoder reranking? (yes/no): "
).strip().lower()

if use_reranking == "yes":

    candidate_k = int(
        input("How many candidates to retrieve before reranking? (try 5): ")
    )

    final_k = int(
        input("How many chunks to keep after reranking? (try 2 or 3): ")
    )

    # Retrieve more candidates
    retriever = index.as_retriever(
        similarity_top_k=candidate_k
    )

    candidate_nodes = retriever.retrieve(question)

    # Build question/chunk pairs
    pairs = [
        [question, node.node.get_content()]
        for node in candidate_nodes
    ]

    # Cross-Encoder scores
    rerank_scores = reranker.predict(pairs)

    reranked_results = list(
        zip(candidate_nodes, rerank_scores)
    )

    # Sort highest reranker score first
    reranked_results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    final_nodes = [
        item[0]
        for item in reranked_results[:final_k]
    ]

else:

    final_k = top_k
    candidate_nodes = retrieved_nodes
    final_nodes = retrieved_nodes

print("\nFinal number of chunks:", len(final_nodes))

RERANKING
Use Cross-Encoder reranking? (yes/no): yes
How many candidates to retrieve before reranking? (try 5): 5
How many chunks to keep after reranking? (try 2 or 3): 3

Final number of chunks: 3


In [ ]:
print("\nVECTOR RETRIEVAL RESULTS")
print("=" * 80)

for rank, node in enumerate(candidate_nodes, start=1):

    print(f"\nVector Rank: {rank}")
    print(f"Vector Similarity: {node.score:.4f}")

    print(node.node.get_content()[:500])

    print("-" * 80)


if use_reranking == "yes":

    print("\n\nCROSS-ENCODER RERANKED RESULTS")
    print("=" * 80)

    for rank, (node, rerank_score) in enumerate(
        reranked_results,
        start=1
    ):

        print(f"\nReranker Rank: {rank}")

        print(
            f"Original Vector Score: "
            f"{node.score:.4f}"
        )

        print(
            f"Cross-Encoder Score: "
            f"{rerank_score:.4f}"
        )

        print(node.node.get_content()[:500])

        print("-" * 80)


VECTOR RETRIEVAL RESULTS

Vector Rank: 1
Vector Similarity: 0.5726
ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or t
--------------------------------------------------------------------------------

Vector Rank: 2
Vector Similarity: 0.4385
Employees working remotely must remain available during core working hours
from 10 AM to 4 PM.
Employees may temporarily work remotely for five days per week when approved
by both their manager and department head.

Parental Leave:
Employees who have completed at least twelve months of continuous service
ar

In [ ]:
context_parts = []

for i, node in enumerate(final_nodes, start=1):

    chunk_text = node.node.get_content()

    context_parts.append(
        f"[CHUNK {i}]\n{chunk_text}"
    )

context = "\n\n".join(context_parts)

print("FINAL CONTEXT SENT TO LLM")
print("=" * 80)

print(context)

FINAL CONTEXT SENT TO LLM
[CHUNK 1]
ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or two days of sick leave, a medical certificate is normally not required.
If an employee takes more than two consecutive days of sick leave,
the employee must submit a medical certificate.
Unused sick leave cannot be carried forward to the next calendar year.

Remote Work:
Employees may work remotely for up to three days per week.
Remote work requires approval from the employee's reporting manager.
Employees working remotely must remain available during core work

In [ ]:
llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0
)

system_prompt = """
You are an HR policy assistant.

Answer the user's question using ONLY the supplied context.

Rules:

1. Do not use outside knowledge.
2. If the context does not contain enough information,
   clearly say what information is missing.
3. Mention all relevant requirements, approvals,
   limits and documentation.
4. Do not invent company policies.
"""

augmented_prompt = f"""
CONTEXT:

{context}


USER QUESTION:

{question}


Answer the question based only on the context above.
"""

print("AUGMENTED PROMPT")
print("=" * 80)

print(augmented_prompt)

AUGMENTED PROMPT

CONTEXT:

[CHUNK 1]
ACME Technologies Employee Leave Policy

Annual Leave:
Every full-time employee receives 20 days of annual leave per calendar year.
Annual leave should normally be requested at least 7 days in advance.
Managers may reject leave requests during critical project delivery periods.
Unused annual leave can be carried forward to the next year.
However, employees can carry forward a maximum of 5 unused annual leave days.

Sick Leave:
Employees receive 10 days of paid sick leave every year.
For one or two days of sick leave, a medical certificate is normally not required.
If an employee takes more than two consecutive days of sick leave,
the employee must submit a medical certificate.
Unused sick leave cannot be carried forward to the next calendar year.

Remote Work:
Employees may work remotely for up to three days per week.
Remote work requires approval from the employee's reporting manager.
Employees working remotely must remain available during core wo

In [ ]:
response = llm.complete(
    augmented_prompt
)

final_answer = response.text

print("FINAL RAG ANSWER")
print("=" * 80)

print(final_answer)

FINAL RAG ANSWER
Every full-time employee receives 20 days of annual leave per calendar year. If you are a full-time employee, you have 20 days of annual leave. However, if you have any unused leave from the previous year, you can carry forward a maximum of 5 unused annual leave days.


In [ ]:
reference_document = DOCUMENT_TEXT

print("Evaluation reference document ready.")

Evaluation reference document ready.


In [ ]:
evaluation_prompt = f"""
You are evaluating a Retrieval-Augmented Generation (RAG) system.

QUESTION:
{question}

FULL SOURCE DOCUMENT:
{reference_document}

RETRIEVED CONTEXT:
{context}

RAG ANSWER:
{final_answer}

Evaluate the system using the following rubric.

1. RETRIEVAL RELEVANCE
How relevant are the retrieved chunks to the user's question?

10 = All retrieved information is highly relevant
7-9 = Mostly relevant with minor unnecessary information
4-6 = Mixed relevant and irrelevant information
1-3 = Mostly irrelevant
0 = Completely irrelevant


2. RETRIEVAL COMPLETENESS
Did retrieval find ALL information from the source document
needed to answer the question?

10 = All necessary information retrieved
7-9 = Almost everything retrieved
4-6 = Important information missing
1-3 = Most required information missing
0 = Retrieval completely failed


3. FAITHFULNESS
Is every factual claim in the answer supported by
the retrieved context?

10 = Completely supported
7-9 = Mostly supported
4-6 = Some unsupported claims
1-3 = Many unsupported claims
0 = Answer contradicts or ignores context


4. ANSWER CORRECTNESS
Compare the answer against the FULL SOURCE DOCUMENT.

10 = Completely correct
7-9 = Mostly correct
4-6 = Partially correct
1-3 = Mostly incorrect
0 = Completely incorrect


5. ANSWER COMPLETENESS
Does the answer fully address everything asked by the user?

10 = Complete answer
7-9 = Minor details missing
4-6 = Important details missing
1-3 = Major omissions
0 = Does not answer the question


Return ONLY valid JSON in this format:

{{
  "retrieval_relevance": <0-10>,
  "retrieval_completeness": <0-10>,
  "faithfulness": <0-10>,
  "answer_correctness": <0-10>,
  "answer_completeness": <0-10>,
  "explanation": {{
      "retrieval_relevance": "...",
      "retrieval_completeness": "...",
      "faithfulness": "...",
      "answer_correctness": "...",
      "answer_completeness": "..."
  }}
}}
"""

In [ ]:
import json

judge_llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0
)

judge_response = judge_llm.complete(
    evaluation_prompt
)

raw_evaluation = judge_response.text.strip()

# Remove markdown fences if the model happens to return them
raw_evaluation = raw_evaluation.replace("```json", "")
raw_evaluation = raw_evaluation.replace("```", "")
raw_evaluation = raw_evaluation.strip()

evaluation = json.loads(raw_evaluation)

print("Evaluation completed.")

Evaluation completed.


In [ ]:
scores = {
    "Retrieval Relevance": evaluation["retrieval_relevance"],
    "Retrieval Completeness": evaluation["retrieval_completeness"],
    "Faithfulness": evaluation["faithfulness"],
    "Answer Correctness": evaluation["answer_correctness"],
    "Answer Completeness": evaluation["answer_completeness"]
}

average_score = sum(scores.values()) / len(scores)

rag_score = average_score * 10

print("=" * 60)
print("RAG EVALUATION")
print("=" * 60)

for metric, score in scores.items():
    print(f"{metric:<25}: {score}/10")

print("-" * 60)

print(f"OVERALL RAG SCORE       : {rag_score:.1f}/100")

RAG EVALUATION
Retrieval Relevance      : 10/10
Retrieval Completeness   : 10/10
Faithfulness             : 10/10
Answer Correctness       : 10/10
Answer Completeness      : 10/10
------------------------------------------------------------
OVERALL RAG SCORE       : 100.0/100


In [ ]:
print("\nEVALUATION EXPLANATION")
print("=" * 80)

for metric, explanation in evaluation["explanation"].items():

    readable_name = metric.replace("_", " ").title()

    print(f"\n{readable_name}")
    print("-" * 40)
    print(explanation)


EVALUATION EXPLANATION

Retrieval Relevance
----------------------------------------
All retrieved information is highly relevant as it directly addresses the user's question about annual leave.

Retrieval Completeness
----------------------------------------
All necessary information regarding annual leave entitlements and carry forward rules was retrieved.

Faithfulness
----------------------------------------
Every factual claim in the answer is supported by the retrieved context, accurately reflecting the source document.

Answer Correctness
----------------------------------------
The answer is completely correct, matching the details provided in the full source document.

Answer Completeness
----------------------------------------
The answer fully addresses everything asked by the user, providing a complete response regarding annual leave.


In [ ]:
results = pd.DataFrame({
    "Metric": list(scores.keys()),
    "Score / 10": list(scores.values())
})

results["Percentage"] = results["Score / 10"] * 10

display(results)

print()
print(f"FINAL RAG SCORE: {rag_score:.1f}/100")

,Metric,Score / 10,Percentage
0,Retrieval Relevance,10,100
1,Retrieval Completeness,10,100
2,Faithfulness,10,100
3,Answer Correctness,10,100
4,Answer Completeness,10,100



FINAL RAG SCORE: 100.0/100
